In [12]:
import operator
from langgraph.graph import StateGraph, START, END
from typing import Annotated, List, TypedDict
from langgraph.types import Send
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

In [23]:
fanout_worker_llm = ChatOpenAI(model="gpt-4o-mini")
editor_base_llm = ChatOpenAI(model="gpt-4o")

In [14]:
class OverallState(TypedDict):
    topic: str 
    subtopics: List[str] 
    research_results: Annotated[List[str], operator.add]
    final_report: str 

class ResearchState(TypedDict): # private state
    subtopic: str 

In [15]:
class SubtopicPlan(BaseModel):
    subtopics: List[str] = Field(..., description="List of Subtopics to research")

In [16]:
async def generate_subtopics(state: OverallState) -> dict:
    topic = state['topic']

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Break the given research topic into clear, specific subtopics for a comprehensive report."),
        ("human", "Topic: {topic}"),
    ])

    planner_chain = prompt | fanout_worker_llm.with_structured_output(SubtopicPlan)

    plan: SubtopicPlan = await planner_chain.ainvoke({
        "topic": topic,
    })

    return {
        "subtopics": plan.subtopics
    }

In [17]:
async def research_subtopic(state: ResearchState) -> dict:
    subtopic = state['subtopic']

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Provide a concise, insightful 1-paragraph research summary of the following subtopic with key facts and insights."),
        ("human", "Subtopic: {subtopic}"),
    ])

    researcher_chain = prompt | fanout_worker_llm

    result = await researcher_chain.ainvoke({
        "subtopic": subtopic
    })

    finding = f'### {subtopic} ###\n{result.content}'

    return {
        "research_results": [finding]
    }

In [18]:
async def compile_report(state: OverallState) -> dict:
    combined_notes = "\n\n".join(state["research_results"])

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a senior research editor. Synthesize the following subtopic research notes "
            "into a well-structured, cohesive executive report with an introduction and conclusion.",
        ),
        ("human", "Main Topic: {topic}\nSubtopic Research Notes: {combined_notes}"),
    ])

    editor_chain = prompt | editor_base_llm

    response = await editor_chain.ainvoke({
        "topic": state["topic"],
        "combined_notes": combined_notes
    })

    return {
        "final_report": response.content
    }

In [19]:
def map_to_researcher(state: OverallState):
    return [
        Send(
            "research_subtopic",
            {"subtopic": s}
        )
        for s in state["subtopics"]
    ]

In [20]:
builder = StateGraph(OverallState)

builder.add_node("generate_subtopics", generate_subtopics)
builder.add_node("research_subtopic", research_subtopic)
builder.add_node("compile_report", compile_report)

builder.add_edge(START, "generate_subtopics")
builder.add_conditional_edges("generate_subtopics", map_to_researcher)
builder.add_edge("research_subtopic", "compile_report")
builder.add_edge("compile_report", END)

graph = builder.compile()

In [21]:
initial_state = {
    "topic": "The evolution of GPU in the last 10 years",
    "research_results": [],
    "final_report": "",
    "subtopics": [],
}

In [24]:
result = await graph.ainvoke(initial_state)

In [25]:
print(result)

{'topic': 'The evolution of GPU in the last 10 years', 'subtopics': ['Introduction to GPU: Basics and Historical Context', 'Technological Advancements in GPU Architecture', 'Comparison of Leading GPU Manufacturers (NVIDIA, AMD, Intel)', 'Impact of Machine Learning and AI on GPU Development', 'Evolution of GPU Memory Technologies (GDDR, HBM, etc.)', 'GPU Performance Metrics Over the Last Decade', 'Trends in GPU Cooling and Power Management', 'The Role of GPUs in Gaming: Advancements and Changes', 'GPUs in Non-Gaming Applications: Data Science, AI, and Visualization', 'Future Trends in GPU Technology and Predictions for the Next Decade'], 'research_results': ['### Introduction to GPU: Basics and Historical Context ###\nGraphics Processing Units (GPUs) have evolved significantly since their inception in the late 20th century, originally designed to accelerate rendering for computer graphics. The first dedicated GPUs emerged in the mid-1990s, with companies like NVIDIA and ATI (now part of

In [27]:
print(result["final_report"])

**Executive Report: The Evolution of GPU in the Last 10 Years**

**Introduction**

Graphics Processing Units (GPUs), since their inception in the late 20th century, have undergone a remarkable evolution, transcending their original purpose of accelerating computer graphics rendering. Initially spearheaded by companies like NVIDIA and ATI, now part of AMD, the journey of GPUs began with the introduction of dedicated hardware designed to handle complex visual tasks more efficiently than central processing units (CPUs). Over the past decade, GPUs have evolved significantly in response to increasing computational demands across a wide range of applications, including gaming, scientific computing, and machine learning, reflecting a synergy between hardware innovation and diverse computational uses. This report examines the technological advancements, competitive landscape, application expansions, and future trends that have characterized the evolution of GPUs over the last ten years.

**Tec